In [ ]:
import osmnx as ox
from osmnx.features import features_from_bbox
import geopandas as gpd
from src.feature_building_utils import *
from src.geometric_utils import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import toml

In [ ]:
df = pd.read_parquet('data/processed_data/S3-approx-coordinates.parquet')

In [ ]:
docs = toml.load("documentation/feature_docs.toml")

In [ ]:
# a) building footprints (+ any height/levels tags)
tags_buildings = {"building": True}
gdf_buildings = features_from_bbox(BBOX, tags_buildings)

In [ ]:
gdf_buildings = gdf_buildings.replace({"building:levels": {"piano terra": 0}})
gdf_buildings.loc[gdf_buildings['level']==str(-1), 'level'] = np.nan
gdf_buildings['level'] = gdf_buildings['level'].astype(float)
gdf_buildings["building:levels"] = gdf_buildings["building:levels"].astype(float)
gdf_buildings["building:levels"] = gdf_buildings["building:levels"].fillna(gdf_buildings["level"])
gdf_buildings.drop(columns=["level"], inplace=True)

In [ ]:
exclude = [
    "geometry",
    "name",
    "name:de",
    "name:es",
    "name:fr",
    "roof:levels",
    "wikidata",
    "wikimedia_commons",
    "wikipedia",
    "contact:street",
    "ref",
    "alt_name",
    "description",
    "operator",
    "website",
    "website",
    "name:en",
    "name:it",
    "phone",
    "addr:housenumber",
    "loc_name",
    "short_name",
    "ref:vatin",
    "source",
    "start_date",
    "addr:unit"
    "end_date",
    "note",
    "internet_access",
    "internet_access:ssid",
    "internet_access:fee",
    "opening_hours",
    "ref:isil",
    "check_date",
    "check_date:opening_hours",
    "opening_hours:signed",
    "contact:email",
    "contact:website",
    "old_name",
    "contact:phone",
    "fax",
    "operator:wikipedia",
    "operator:wikidata"
    ]
gdf_imputed = impute_gdf(gdf_buildings, exclude=exclude, max_rank=50)

In [ ]:
feature_name = "close2industrial_building_15"

df[feature_name] = df.apply(
    lambda row: is_close_to(
        features=gdf_buildings,
        point=Point(row.x, row.y),
        threshold=15,
        type_column="building",
        types=["industrial"],
    ), axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 15m away from an industrial building"
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {0, 1}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2industrial_building_100"

df[feature_name] = df.apply(
    lambda row: is_close_to(
        features=gdf_buildings,
        point=Point(row.x, row.y),
        threshold=100,
        type_column="building",
        types=["industrial"],
    ), axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 100m away from an industrial building"
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {0, 1}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "average_building_height_100"

df[feature_name] = df.apply(
    lambda row: get_nearest_rows(
        gdf_imputed,
        Point(row.x, row.y),
        radius_meters=100
    )['building:levels'].mean(), axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Average (low-rank SVD) estimated building height within a 100m radius circle around the point"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {float(df[feature_name].min()), float(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_residential_buildings_200"

df[feature_name] = df.apply(
    lambda row: count_nearby(
        gdf_buildings,
        Point(row.x, row.y),
        threshold=200,
        type_column="building",
        types=["residential", "apartments", "house", "dormitory", "semidetached_house"]
    ), axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of residential buildings within a 200m radius circle around the point"
docs[feature_name]["type"] = "integer"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "proportion_residential_buildings_500"

df[feature_name] = df.apply(
    lambda row: land_cover_proportion(
        gdf_buildings,
        Point(row.x, row.y),
        threshold=500,
        type_column="building",
        types=["residential", "apartments", "house", "dormitory", "semidetached_house"]
    ), axis=1
).astype(float)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Proportion of residential buildings within a 500m radius circle around the point"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {float(df[feature_name].min()), float(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2commercial_building_25"

df[feature_name] = df.apply(
    lambda row: is_close_to(
        features=gdf_buildings,
        point=Point(row.x, row.y),
        threshold=25,
        type_column="building",
        types=["commercial"],
    ), axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 25m away from a commercial building"
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {0, 1}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_commercial_buildings_200"

df[feature_name] = df.apply(
    lambda row: count_nearby(
        gdf_buildings,
        Point(row.x, row.y),
        threshold=200,
        type_column="building",
        types=["commercial", "service", "office", "kiosk", "retail"]
    ), axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of commercial buildings within a 200m radius circle around the point"
docs[feature_name]["type"] = "integer"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_buildings.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
with open("documentation/feature_docs.toml", "w") as f:
    toml.dump(docs, f)

In [ ]:
df.to_parquet('data/processed_data/S3-approx-coordinates.parquet')